In [ ]:
!pip install -q news-please warcio pyarrow

In [ ]:
import os
import re
import io
import json
import time
import random
import hashlib
import gzip
from pathlib import Path
from typing import List, Dict, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm
from newsplease import NewsPlease
from warcio.archiveiterator import ArchiveIterator

In [ ]:
# ---------- USER CONFIG ----------
# temp for testing
PRE_TARGET = 20000
POST_TARGET = 20000

PRE_START_DATE = "2018-01-01"
PRE_END_DATE   = "2021-12-31"

POST_START_DATE = "2022-01-01"
POST_END_DATE   = "2026-04-02"

MAX_PER_DOMAIN = 2500
MAX_WARC_FILES_PER_MONTH = 20
MAX_ARTICLES_PER_WARC = 100
MIN_WORD_COUNT = 128
REQUEST_SLEEP_RANGE = (0.05, 0.15)
MAX_ARTICLES_PER_MONTH = 420

# Restrict to a curated publisher list.
# To remove publisher restrictions later, set DOMAINS = None
DOMAINS = [
    "nytimes.com",
    "washingtonpost.com",
    "wsj.com",
    "usatoday.com",
    "latimes.com",
    "chicagotribune.com",
    "bostonglobe.com",
    "sfchronicle.com",
    "seattletimes.com",
    "cnn.com",
    "nbcnews.com",
    "abcnews.go.com",
    "cbsnews.com",
    "npr.org",
    "pbs.org",
    "politico.com",
    "thehill.com",
    "axios.com",
    "bbc.com",
    "bbc.co.uk",
    "reuters.com",
    "apnews.com",
    "theguardian.com",
    "ft.com",
    "economist.com",
    "independent.co.uk",
    "bloomberg.com",
    "businessinsider.com",
    "forbes.com",
    "time.com",
    "theatlantic.com",
    "newsweek.com",
    "huffpost.com",
    "marketwatch.com",
    "fortune.com",
    "yahoo.com",
    "news.yahoo.com",
    "cnbc.com",
    "foxnews.com",
    "msnbc.com",
    "newsweek.com",
    "usnews.com",
    "aljazeera.com",
    "dw.com",
    "telegraph.co.uk",
    "nypost.com",
    "thetimes.co.uk",
    "scmp.com",
    "japantimes.co.jp",
    "straitstimes.com",
    "irishtimes.com",
    "globeandmail.com",
    "nationalpost.com",
    "cbc.ca",
    "ctvnews.ca",
    "torontosun.com",
    "denverpost.com",
    "miamiherald.com",
    "startribune.com",
    "philly.com",
    "inquirer.com",
    "newsobserver.com",
    "houstonchronicle.com",
    "dallasnews.com",
]

# Local save location
OUTPUT_DIR = Path("news_dataset_output")

PRE_DIR = OUTPUT_DIR / "pre_2022"
POST_DIR = OUTPUT_DIR / "post_2021"

for base in [PRE_DIR, POST_DIR]:
    (base / "raw_jsonl").mkdir(parents=True, exist_ok=True)
    (base / "parquet").mkdir(parents=True, exist_ok=True)
    (base / "csv").mkdir(parents=True, exist_ok=True)

In [ ]:
def safe_getattr(obj, attr, default=None):
    try:
        value = getattr(obj, attr, default)
        return value if value is not None else default
    except Exception:
        return default

def text_hash(text: str) -> str:
    if not isinstance(text, str):
        text = ""
    return hashlib.sha256(text.strip().encode("utf-8", errors="ignore")).hexdigest()

In [ ]:
def clean_scraped_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df[
        (df["success"] == True) &
        (df["maintext"].fillna("").str.len() > 0) &
        (df["word_count"].fillna(0) >= MIN_WORD_COUNT)
    ].copy()

    if "text_hash" in df.columns:
        df = df.drop_duplicates(subset=["text_hash"])

    if "url" in df.columns:
        df = df.drop_duplicates(subset=["url"])

    return df.reset_index(drop=True)


def month_windows(start_date: str, end_date: str):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    current = start.replace(day=1)

    while current <= end:
        next_month = current + pd.offsets.MonthBegin(1)
        window_start = max(current, start)
        window_end = min(next_month - pd.Timedelta(days=1), end)

        yield window_start.strftime("%Y-%m-%d"), window_end.strftime("%Y-%m-%d")
        current = next_month


def get_ccnews_warc_listing_url(year: int, month: int) -> str:
    return f"https://data.commoncrawl.org/crawl-data/CC-NEWS/{year}/{month:02d}/warc.paths.gz"


def fetch_ccnews_warc_paths(year: int, month: int, max_warc_files: int = 25) -> List[str]:
    listing_url = get_ccnews_warc_listing_url(year, month)

    resp = requests.get(listing_url, timeout=60)
    resp.raise_for_status()

    warc_paths = []
    with gzip.GzipFile(fileobj=io.BytesIO(resp.content)) as gz:
        for line in gz:
            path = line.decode("utf-8", errors="ignore").strip()
            if path:
                warc_paths.append("https://data.commoncrawl.org/" + path)
            if len(warc_paths) >= max_warc_files:
                break

    return warc_paths


def extract_article_from_html(url: str, html: str, crawl_date: Optional[str] = None) -> Dict:
    try:
        article = NewsPlease.from_html(html, url=url)

        if article is None:
            return {
                "url": url,
                "success": False,
                "error": "newsplease_from_html_returned_none"
            }

        maintext = safe_getattr(article, "maintext", "")
        title = safe_getattr(article, "title", "")
        description = safe_getattr(article, "description", "")
        date_publish = safe_getattr(article, "date_publish", None)
        authors = safe_getattr(article, "authors", None)
        language = safe_getattr(article, "language", None)
        source_domain = safe_getattr(article, "source_domain", None)

        if date_publish is None and crawl_date is not None:
            date_publish = crawl_date

        word_count = len(maintext.split()) if isinstance(maintext, str) and maintext.strip() else 0

        return {
            "url": url,
            "success": True,
            "title": title,
            "description": description,
            "maintext": maintext,
            "date_publish": str(date_publish) if date_publish is not None else None,
            "authors": authors,
            "language": language,
            "source_domain": source_domain,
            "word_count": word_count,
            "text_hash": text_hash(maintext),
        }

    except Exception as e:
        return {
            "url": url,
            "success": False,
            "error": repr(e)
        }


def extract_articles_from_ccnews_warc(
    warc_url: str,
    domains: Optional[List[str]] = None,
    max_articles: int = 100
) -> List[Dict]:
    results = []

    try:
        resp = requests.get(warc_url, stream=True, timeout=120)
        resp.raise_for_status()

        gz = gzip.GzipFile(fileobj=resp.raw)

        for record in ArchiveIterator(gz):
            if len(results) >= max_articles:
                break

            if record.rec_type != "response":
                continue

            url = record.rec_headers.get_header("WARC-Target-URI")
            crawl_date = record.rec_headers.get_header("WARC-Date")

            if not url or not url.startswith("http"):
                continue

            if domains and not any(domain in url for domain in domains):
                continue

            http_headers = getattr(record, "http_headers", None)
            if http_headers is None:
                continue

            statusline = getattr(http_headers, "statusline", "") or ""
            if "200" not in statusline:
                continue

            content_type = http_headers.get_header("Content-Type") or ""
            if "html" not in content_type.lower():
                continue

            try:
                html_bytes = record.content_stream().read()
                html = html_bytes.decode("utf-8", errors="ignore")
            except Exception:
                continue

            result = extract_article_from_html(url=url, html=html, crawl_date=crawl_date)
            results.append(result)

    except Exception as e:
        print(f"Failed to process WARC file {warc_url}: {e}")

    return results


def get_bucket_dir(label: str) -> Path:
    if label == "pre_2022":
        return PRE_DIR
    elif label == "post_2021":
        return POST_DIR
    else:
        raise ValueError(f"Unknown label: {label}")


def get_month_file_path(label: str, month_key: str) -> Path:
    bucket_dir = get_bucket_dir(label)
    return bucket_dir / f"{label}_{month_key}.parquet"


def get_completed_months_from_drive(label: str) -> set:
    bucket_dir = get_bucket_dir(label)
    completed = set()

    for path in bucket_dir.glob(f"{label}_*.parquet"):
        stem = path.stem  # e.g. pre_2022_2018-01
        month_key = stem.replace(f"{label}_", "")
        completed.add(month_key)

    return completed


def load_existing_bucket_from_drive(label: str) -> pd.DataFrame:
    bucket_dir = get_bucket_dir(label)
    files = sorted(bucket_dir.glob(f"{label}_*.parquet"))

    dfs = []
    for path in files:
        try:
            df = pd.read_parquet(path)
            dfs.append(df)
        except Exception as e:
            print(f"[{label}] Failed to read {path}: {e}")

    if len(dfs) == 0:
        return pd.DataFrame()

    combined = pd.concat(dfs, ignore_index=True)

    if "url" in combined.columns:
        combined = combined.drop_duplicates(subset=["url"])
    if "text_hash" in combined.columns:
        combined = combined.drop_duplicates(subset=["text_hash"])

    return combined.reset_index(drop=True)


def collect_bucket(
    label: str,
    start_date: str,
    end_date: str,
    target_n: int,
    domains: Optional[List[str]] = None,
    max_warc_files_per_month: int = 25,
    max_articles_per_warc: int = 100,
    max_articles_per_month: Optional[int] = None,
    max_per_domain_per_month: Optional[int] = None,
    sleep_range=(1.0, 2.0),
):
    # Load existing monthly files from Drive
    existing_df = load_existing_bucket_from_drive(label)

    if len(existing_df) > 0:
        all_rows = existing_df.to_dict(orient="records")
        seen_urls = set(existing_df["url"].dropna().tolist()) if "url" in existing_df.columns else set()
        seen_text_hashes = set(existing_df["text_hash"].dropna().tolist()) if "text_hash" in existing_df.columns else set()

        if "source_domain" in existing_df.columns:
            domain_counts = existing_df["source_domain"].value_counts().to_dict()
        else:
            domain_counts = {}

        completed_months = get_completed_months_from_drive(label)

        print(f"[{label}] Loaded {len(existing_df)} existing rows from Drive")
        print(f"[{label}] Completed months already on Drive: {len(completed_months)}")

        if len(all_rows) >= target_n:
            final_df = pd.DataFrame(all_rows).iloc[:target_n].reset_index(drop=True)
            print(f"[{label}] Target already met from Drive data")
            return final_df

    else:
        all_rows = []
        seen_urls = set()
        seen_text_hashes = set()
        domain_counts = {}
        completed_months = set()

    # Iterate month by month
    for window_start, window_end in month_windows(start_date, end_date):
        if len(all_rows) >= target_n:
            break

        month_key = window_start[:7]  # YYYY-MM

        if month_key in completed_months:
            print(f"[{label}] Skipping {month_key} (already saved to Drive)")
            continue

        print(f"[{label}] Processing month {month_key}")

        year = int(window_start[:4])
        month = int(window_start[5:7])

        try:
            warc_urls = fetch_ccnews_warc_paths(
                year=year,
                month=month,
                max_warc_files=max_warc_files_per_month
            )
        except Exception as e:
            print(f"[{label}] CC-NEWS listing failed for {month_key}: {e}")
            continue

        if len(warc_urls) == 0:
            print(f"[{label}] No WARC files found for {month_key}")
            continue

        print(f"[{label}] WARC files this month: {len(warc_urls)}")

        month_rows = []

        for i, warc_url in enumerate(warc_urls, start=1):
            print(f"[{label}]   Processing WARC {i}/{len(warc_urls)}")

            try:
                warc_results = extract_articles_from_ccnews_warc(
                    warc_url=warc_url,
                    domains=domains,
                    max_articles=max_articles_per_warc
                )
            except Exception as e:
                print(f"[{label}]   Failed WARC {i}: {e}")
                continue

            print(f"[{label}]   Extracted {len(warc_results)} candidate records from this WARC")

            if len(warc_results) == 0:
                continue

            month_rows.extend(warc_results)
            time.sleep(random.uniform(*sleep_range))

        if len(month_rows) == 0:
            print(f"[{label}] No candidate articles found for {month_key}")
            continue

        scraped_df = pd.DataFrame(month_rows)
        clean_df = clean_scraped_df(scraped_df)

        if len(clean_df) == 0:
            print(f"[{label}] No clean articles survived filtering for {month_key}")
            continue

        clean_df = clean_df[~clean_df["url"].isin(seen_urls)].copy()
        clean_df = clean_df[~clean_df["text_hash"].isin(seen_text_hashes)].copy()

        if len(clean_df) == 0:
            print(f"[{label}] All clean articles in {month_key} were duplicates")
            continue

        clean_df["time_bucket"] = label
        clean_df["month_key"] = month_key

        filtered_rows = []
        month_domain_counts = {}

        for _, row in clean_df.iterrows():
            domain = row.get("source_domain")

            if domain is None:
                continue

            global_count = domain_counts.get(domain, 0)
            month_count = month_domain_counts.get(domain, 0)

            if global_count >= MAX_PER_DOMAIN:
                continue

            if max_per_domain_per_month is not None and month_count >= max_per_domain_per_month:
                continue

            filtered_rows.append(row)
            domain_counts[domain] = global_count + 1
            month_domain_counts[domain] = month_count + 1

        if len(filtered_rows) == 0:
            print(f"[{label}] All clean articles in {month_key} exceeded per-domain limits")
            continue

        filtered_df = pd.DataFrame(filtered_rows)

        if max_articles_per_month is not None and len(filtered_df) > max_articles_per_month:
            filtered_df = filtered_df.sample(
                n=max_articles_per_month,
                random_state=42
            ).reset_index(drop=True)

        remaining_needed = target_n - len(all_rows)
        if len(filtered_df) > remaining_needed:
            filtered_df = filtered_df.iloc[:remaining_needed].reset_index(drop=True)

        if len(filtered_df) == 0:
            print(f"[{label}] No rows remained after final month trimming for {month_key}")
            continue

        month_file_path = get_month_file_path(label, month_key)
        filtered_df.to_parquet(month_file_path, index=False)

        print(f"[{label}] Saved {len(filtered_df)} rows for {month_key} to {month_file_path}")

        for u in filtered_df["url"].dropna().tolist():
            seen_urls.add(u)

        for h in filtered_df["text_hash"].dropna().tolist():
            seen_text_hashes.add(h)

        all_rows.extend(filtered_df.to_dict(orient="records"))
        completed_months.add(month_key)

        print(f"[{label}] Running total: {len(all_rows)} / {target_n}")

        if "source_domain" in filtered_df.columns:
            print(filtered_df["source_domain"].value_counts().head(10))

    final_df = pd.DataFrame(all_rows)

    if len(final_df) > 0:
        if "url" in final_df.columns:
            final_df = final_df.drop_duplicates(subset=["url"])
        if "text_hash" in final_df.columns:
            final_df = final_df.drop_duplicates(subset=["text_hash"])
        final_df = final_df.iloc[:target_n].reset_index(drop=True)

    print(f"[{label}] Final total loaded/saved across monthly Drive files: {len(final_df)}")
    return final_df


def save_bucket(df: pd.DataFrame, base_dir: Path, run_name: str):
    raw_dir = base_dir / "raw_jsonl"
    parquet_dir = base_dir / "parquet"
    csv_dir = base_dir / "csv"

    jsonl_path = raw_dir / f"{run_name}.jsonl"
    csv_path = csv_dir / f"{run_name}.csv"
    parquet_path = parquet_dir / f"{run_name}.parquet"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for record in df.to_dict(orient="records"):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)

    print(f"Saved {len(df)} rows to:")
    print(" -", jsonl_path.resolve())
    print(" -", csv_path.resolve())
    print(" -", parquet_path.resolve())

In [ ]:
import logging
import warnings
from dateutil.parser import UnknownTimezoneWarning

# Silence very noisy library logs
logging.getLogger("newspaper").setLevel(logging.ERROR)
logging.getLogger("jieba").setLevel(logging.ERROR)

# Silence warnings that are not breaking the pipeline
warnings.filterwarnings("ignore", category=UnknownTimezoneWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/news_data")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

PRE_DIR = DRIVE_ROOT / "pre_2022"
POST_DIR = DRIVE_ROOT / "post_2021"

PRE_DIR.mkdir(parents=True, exist_ok=True)
POST_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pre_df = collect_bucket(
    label="pre_2022",
    start_date=PRE_START_DATE,
    end_date=PRE_END_DATE,
    target_n=PRE_TARGET,
    domains=DOMAINS,
    max_warc_files_per_month=MAX_WARC_FILES_PER_MONTH,
    max_articles_per_warc=MAX_ARTICLES_PER_WARC,
    max_articles_per_month=MAX_ARTICLES_PER_MONTH,
    sleep_range=REQUEST_SLEEP_RANGE,
)

In [ ]:
post_df = collect_bucket(
    label="post_2021",
    start_date=POST_START_DATE,
    end_date=POST_END_DATE,
    target_n=POST_TARGET,
    domains=DOMAINS,
    max_warc_files_per_month=MAX_WARC_FILES_PER_MONTH,
    max_articles_per_warc=MAX_ARTICLES_PER_WARC,
    max_articles_per_month=400,
    sleep_range=REQUEST_SLEEP_RANGE,
)